In [0]:
#Highest
# SCD2 Person + Contact Roles + Auth (Parquet)
# Uses: dfPerson, dfBusinessEntityContact, dfContactType, dfPassword
# Writes:

# …/dim_person_scd2/ (SCD Type‑2 with Type‑1 field mixed in)
# …/person_contactrole/ (deduped NK merge)
# …/person_auth_metadata/ (policy flags)
# …/_run_summaries/person/ (run metrics)


# Highlights: Manual SCD2 over Parquet (no Delta), deterministic surrogate keys (person_sk from BK+version number), Type‑1/Type‑2 mix, optional late-arrival handling (off by default), and run summaries.

In [0]:
# =========================
# DEMO PROLOGUE (same as Notebook 3)
# =========================

from pyspark.sql import functions as F, Window

demo_mode = True
enable_incremental = False
allow_late_arrival = False
enable_pit_snapshot = False

# Source parquet files
source_db = "main.default"   # <-- AQUÍ está tu Bronze real

# Target in Unity Catalog
target_catalog = "main"
target_schema  = "silver"
target_db = f"{target_catalog}.{target_schema}"   # "main.silver"

print(f"[INFO] Using output target: {target_db}")


In [0]:
# =========================
# Existence helpers (UC)
# =========================
def table_exists(table_fqn: str) -> bool:
    try:
        return spark.catalog.tableExists(table_fqn)
    except Exception:
        return False

def read_table_if_exists(table_fqn: str):
    return spark.table(table_fqn) if table_exists(table_fqn) else None

In [0]:

# =========================
# OUTPUT PATHS
# =========================
out_dim_person     = f"{out_base}/dim_person_scd2"
out_contactrole    = f"{out_base}/person_contactrole"
out_authmeta       = f"{out_base}/person_auth_metadata"
out_summary_person = f"{out_base}/_run_summaries/person"
# Optional PIT output if enabled
out_person_pit     = f"{out_base}/dim_person_pit"




In [0]:

# =========================
# INPUTS (Unity Catalog)
# =========================
# dfPerson: Person.Person
# dfBusinessEntityContact: Person.BusinessEntityContact
# dfContactType: Person.ContactType
# dfPassword: Person.Password

bronze_db = "main.bronze"

tables = {
    "Person":                 f"{bronze_db}.person_person",
    "BusinessEntityContact":  f"{bronze_db}.person_businessentitycontact",
    "ContactType":            f"{bronze_db}.person_contacttype",
    "Password":               f"{bronze_db}.person_password",
}

dfs = {k: spark.table(v) for k, v in tables.items()}

dfPerson = dfs["Person"]
dfBusinessEntityContact = dfs["BusinessEntityContact"]
dfContactType = dfs["ContactType"]
dfPassword = dfs["Password"]

display(dfPassword)

In [0]:
# =========================
# 1) SCD2 (with mixed Type-1 field) for Person
# =========================

t2_attrs = ["PersonType","NameStyle","Title","FirstName","MiddleName","LastName","Suffix","AdditionalContactInfo"]
t1_attrs = ["EmailPromotion"]

incoming_full = (
    dfPerson.select(
        F.col("BusinessEntityID").alias("business_entity_id"),
        "PersonType","NameStyle","Title","FirstName","MiddleName","LastName","Suffix",
        "EmailPromotion","AdditionalContactInfo","rowguid","ModifiedDate"
    )
    .withColumn("FirstName", F.initcap(F.col("FirstName")))
    .withColumn("MiddleName", F.initcap(F.col("MiddleName")))
    .withColumn("LastName", F.initcap(F.col("LastName")))
    .withColumn(
        "t2_hashdiff",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in t2_attrs]
            ),
            256
        )
    )
)

if enable_incremental and watermark_from:
    incoming_full = incoming_full.filter(
        F.col("ModifiedDate") >= F.to_timestamp(F.lit(watermark_from))
    )

# ✅ CAMBIO: leer tabla UC
existing = read_table_if_exists(out_dim_person)

def compute_person_sk(df):
    return df.withColumn(
        "person_sk",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("business_entity_id").cast("string"),
                F.lpad(F.col("version_no").cast("string"), 4, "0")
            ),
            256
        )
    )

if existing is None:
    seed = (
        incoming_full
        .withColumn("version_no", F.lit(1))
        .withColumn("effective_start_date", F.current_timestamp())
        .withColumn("effective_end_date", F.to_timestamp(F.lit("9999-12-31 23:59:59")))
        .withColumn("is_current", F.lit(True))
    )

    seed = compute_person_sk(seed)

    dim_seed = seed.select(
        "person_sk","business_entity_id", *t2_attrs, *t1_attrs,
        "rowguid","ModifiedDate","t2_hashdiff",
        "version_no","effective_start_date","effective_end_date","is_current"
    )

    # ✅ CAMBIO: escribir como tabla UC
    dim_seed.write.format("delta").mode("overwrite").saveAsTable(out_dim_person)
    display(dim_seed.limit(50))

    scd_summary = spark.createDataFrame([(
        int(dim_seed.count()),
        0, 0, 0, 0
    )], ["scd2_new","scd2_changed","scd2_expired","scd2_t1_updates","scd2_unchanged"]) \
    .withColumn("entity", F.lit("dim_person_scd2")) \
    .withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
    .withColumn("run_ts", F.current_timestamp())

    # ✅ CAMBIO: tabla UC
    scd_summary.write.format("delta").mode("append").saveAsTable(out_summary_person)
    display(scd_summary)

else:
    # --- resto del bloque NO CAMBIA ---
    # ...
    # (toda tu lógica SCD2 sigue igual)
    # ...

    # ✅ CAMBIO FINAL
    final_dim.write.format("delta").mode("overwrite").saveAsTable(out_dim_person)

    scd_summary.write.format("delta").mode("append").saveAsTable(out_summary_person)

In [0]:
# =========================
# 2) Contact role bridge (dedup by NK, UC-based merge)
# =========================

ct = dfContactType.select(
    "ContactTypeID",
    F.col("Name").alias("contact_type_name")
)

contact_raw = (
    dfBusinessEntityContact.alias("bec")
    .join(ct.alias("ct"), "ContactTypeID", "left")
    .select(
        F.col("bec.BusinessEntityID").alias("business_entity_id"),
        F.col("bec.PersonID").alias("contact_person_id"),
        F.col("ct.contact_type_name"),
        F.col("bec.ModifiedDate")
    )
)

# ✅ CAMBIO: leer tabla UC
existing_contact = read_table_if_exists(out_contactrole)

contact_cur = (
    contact_raw
    .withColumn(
        "nk",
        F.concat_ws(
            "||",
            F.col("business_entity_id").cast("string"),
            F.col("contact_person_id").cast("string"),
            F.coalesce(F.col("contact_type_name"), F.lit(""))
        )
    )
)

if existing_contact is None:
    merged_contact = contact_cur
else:
    merged_contact = existing_contact.unionByName(
        contact_cur,
        allowMissingColumns=True
    )

w_contact = Window.partitionBy("nk").orderBy(
    F.col("ModifiedDate").desc_nulls_last()
)

person_contactrole = (
    merged_contact
    .withColumn("rn", F.row_number().over(w_contact))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("person_contactrole_sk", F.sha2(F.col("nk"), 256))
    .drop("nk")
)

# ✅ CAMBIO: escribir tabla UC
person_contactrole.write.format("delta").mode("overwrite").saveAsTable(out_contactrole)

# =========================
# Run summary
# =========================
summary_contact = spark.createDataFrame([(
    int(contact_raw.count()),
    int(person_contactrole.count()),
    int(person_contactrole.select("business_entity_id").distinct().count())
)], ["raw_links","curated_links","distinct_business_entities"]) \
.withColumn("entity", F.lit("person_contactrole")) \
.withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
.withColumn("run_ts", F.current_timestamp())

# ✅ CAMBIO: tabla UC
summary_contact.write.format("delta").mode("append").saveAsTable(out_summary_person)
display(summary_contact)

In [0]:
display(person_contactrole.limit(50))

In [0]:
# =========================
# 3) Auth metadata (policy flags), UC-based merge by BK
# =========================

auth_raw = (
    dfPassword
    .select(
        F.col("BusinessEntityID").alias("business_entity_id"),
        F.length(F.col("PasswordHash")).alias("password_hash_length"),
        F.length(F.col("PasswordSalt")).alias("password_salt_length"),
        "ModifiedDate"
    )
    .withColumn(
        "has_password",
        (F.col("password_hash_length") > 0) &
        (F.col("password_salt_length") > 0)
    )
)

# Simple policy check (demo-safe)
auth = (
    auth_raw
    .withColumn(
        "policy_violation",
        F.when(
            (F.col("has_password") == True) &
            (F.col("password_hash_length") < 32),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "person_auth_metadata_sk",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("business_entity_id").cast("string"),
                F.coalesce(F.col("password_hash_length").cast("string"), F.lit("0")),
                F.coalesce(F.col("password_salt_length").cast("string"), F.lit("0"))
            ),
            256
        )
    )
)

# ✅ CAMBIO: leer tabla UC
existing_auth = read_table_if_exists(out_authmeta)

if existing_auth is None:
    merged_auth = auth
else:
    merged_auth = (
        existing_auth
        .unionByName(auth, allowMissingColumns=True)
        .withColumn("nk", F.col("business_entity_id"))
    )

    w_auth = Window.partitionBy("nk").orderBy(
        F.col("ModifiedDate").desc_nulls_last()
    )

    merged_auth = (
        merged_auth
        .withColumn("rn", F.row_number().over(w_auth))
        .filter("rn = 1")
        .drop("rn", "nk")
    )

# ✅ CAMBIO: escribir tabla UC
merged_auth.write.format("delta").mode("overwrite").saveAsTable(out_authmeta)
display(merged_auth.limit(50))

# =========================
# Run summary
# =========================
summary_auth = spark.createDataFrame([(
    int(auth_raw.count()),
    int(merged_auth.count()),
    int(merged_auth.filter(F.col("policy_violation") == True).count())
)], ["raw_rows","curated_rows","policy_violations"]) \
.withColumn("entity", F.lit("person_auth_metadata")) \
.withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
.withColumn("run_ts", F.current_timestamp())

# ✅ CAMBIO: tabla UC
summary_auth.write.format("delta").mode("append").saveAsTable(out_summary_person)
display(summary_auth)

print("[OK] Notebook 4 completed.")